# 06 · Rank-shift stability & decomposition

Standardizing the pipeline reorders the leaderboard (paper Table 5). Two
questions follow: **(1)** are the shifts real or sampling noise? — an item-level
**bootstrap**; and **(2)** is the shift driven by the LLM judge or by the other
standardizations? — a **generation-vs-extraction decomposition**. Both re-score
stored outputs; no new generation.


In [1]:
import nbtools as nb
from nbtools import show_df, show_fig, show_md, run_mod, heavy, REGEN, REPORTS_DIR
print("SAYF_NB_REGEN =", REGEN, "  (heavy GPU/API steps run only when True)")
print("reports dir  :", REPORTS_DIR)

SAYF_NB_REGEN = False   (heavy GPU/API steps run only when True)
reports dir  : /export/home/aberriche/BenchBench/BenchmarkingSecBenchmarks/unified-benchmark-pipeline/analysis/reports


### Item-level bootstrap CIs — `analysis.bootstrap_ci`  *(heavy · re-scores stored per-item outputs)*
Rendered from cached `bootstrap_ci.json`. `SAYF_NB_REGEN=1` recomputes (5,000
paired resamples of the stored per-item scores). ρ_b is Spearman's ρ between the
original and standardized per-benchmark rankings.


In [2]:
if heavy("bootstrap_ci"):
    import importlib; importlib.import_module("analysis.bootstrap_ci")
import json, pandas as pd
from IPython.display import display
b = json.loads((REPORTS_DIR / "bootstrap_ci.json").read_text())
rows = []
for name, e in b["benchmarks"].items():
    lo, hi = e["rho_ci"]
    rows.append({"benchmark": name, "rho_b": round(e["rho_point"], 2),
                 "ci_low": round(lo, 2), "ci_high": round(hi, 2)})
display(pd.DataFrame(rows))
print(f"{b['B']} resamples; every interval excludes 1, only SecEval includes 0")

[bootstrap_ci] heavy step skipped — rendering cached artifact. Set SAYF_NB_REGEN=1 to recompute (needs GPU/API key).


,benchmark,rho_b,ci_low,ci_high
0,MMLU-CS,-0.47,-0.71,-0.25
1,SecEval,0.03,-0.08,0.15
2,SECURE,0.65,0.56,0.72
3,CTI-Bench,0.64,0.56,0.76
4,AthenaBench,0.42,0.39,0.47
5,CyberMetric,0.73,0.47,0.81
6,RedSage-Bench,0.71,0.60,0.72
7,SecBench,0.50,0.32,0.58


5000 resamples; every interval excludes 1, only SecEval includes 0


### Generation vs. extraction — `analysis.decomp_full`  *(heavy · reads original + standardized generations)*
Rendered from cached `decomp_full.json`. **GEN** = ρ(regex@original → regex@standardized)
(generation change, extractor held at regex); **EXT** = ρ(regex@standardized →
judge@standardized) (extractor swap, generation fixed). Lower ρ = more reordering;
the ρ(A',A) column checks the regex reproduces the published *before* ranking.


In [3]:
if heavy("decomp_full"):
    import importlib; importlib.import_module("analysis.decomp_full")
import json, pandas as pd
from IPython.display import display
d = json.loads((REPORTS_DIR / "decomp_full.json").read_text())
rows = []
for name, e in d["benchmarks"].items():
    rows.append({"benchmark": name,
                 "GEN rho(A'->B')": e["GEN_rho_Ap_Bp"],
                 "EXT rho(B'->C)": e["EXT_rho_Bp_C"],
                 "check rho(A',A)": e["rho_Ap_A_regex_reproduces_before"]})
display(pd.DataFrame(rows))

[decomp_full] heavy step skipped — rendering cached artifact. Set SAYF_NB_REGEN=1 to recompute (needs GPU/API key).


,benchmark,GEN rho(A'->B'),EXT rho(B'->C),"check rho(A',A)"
0,MMLU-CS,0.250000,0.200000,0.908333
1,SecEval,-0.191667,0.272727,0.825000
2,SECURE,0.800000,0.575758,0.775000
3,CTI-Bench,0.591667,0.890909,0.533333
4,AthenaBench,0.341667,0.987879,0.908333
5,CyberMetric,0.933333,0.818182,0.941667
6,RedSage-Bench,0.550000,0.721212,0.841667
7,SecBench,0.383333,0.612121,0.950000


The extraction (judge) swap co-drives the shift only on **MMLU-CS** (EXT ρ≈0.20)
and mildly SECURE; on AthenaBench, CTI-Bench and SecBench the judge barely reorders
(EXT ρ 0.99 / 0.89 / 0.61) and the **generation** change dominates. So Table 5 is
not an artifact of "replacing regexes with a judge" (paper App.~L.4/L.5; the two
attacker-attribution tasks are excluded — a regex cannot do alias matching).
